# 14.9 — Changepoint detection

Changepoint detection asks when one stable rule stopped and another rule began.

Changepoints are persistent anomalies: one segment is better explained by a different mean, slope, or variance. This notebook uses exhaustive tiny split scoring plus CUSUM evidence. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1306
rng = np.random.default_rng(SEED)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def interval_coverage(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    inside = (y_true >= lower) & (y_true <= upper)
    return float(np.mean(inside))


def f13_time_series_ladder():
    t = np.arange(72, dtype=float)
    rungs = []

    y1 = np.full_like(t, 10.0)
    rungs.append({"name": "D1 constant", "t": t, "y": y1, "true": y1, "period": 12})

    y2 = 8.0 + 0.18 * t
    rungs.append({"name": "D2 drift", "t": t, "y": y2, "true": y2, "period": 12})

    noise3 = rng.normal(0.0, 0.45, size=t.size)
    true3 = 8.0 + 0.18 * t
    y3 = true3 + noise3
    rungs.append({"name": "D3 drift + noise", "t": t, "y": y3, "true": true3, "period": 12})

    seasonal4 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true4 = 8.0 + 0.12 * t + seasonal4
    y4 = true4 + rng.normal(0.0, 0.35, size=t.size)
    rungs.append({"name": "D4 seasonal", "t": t, "y": y4, "true": true4, "period": 12})

    seasonal5 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true5 = 8.0 + 0.10 * t + seasonal5
    true5 = true5 + np.where(t >= 48.0, 3.0 + 0.06 * (t - 48.0), 0.0)
    y5 = true5 + rng.normal(0.0, 0.45, size=t.size)
    y5[18] = y5[18] + 5.0
    y5[55] = y5[55] - 4.0
    rungs.append({"name": "D5 outliers + regime shift", "t": t, "y": y5, "true": true5, "period": 12})

    return rungs


def train_test_split_series(rung, train_size=48):
    train = {"t": rung["t"][:train_size], "y": rung["y"][:train_size]}
    test = {"t": rung["t"][train_size:], "y": rung["y"][train_size:], "true": rung["true"][train_size:]}
    return train, test


def print_ladder_preview(rungs):
    for rung in rungs:
        name = rung["name"]
        y = rung["y"]
        sample = np.round(y[:6], 3)
        print(f"{name}: n={y.size}, period={rung['period']}, first six={sample}")


def plot_forecast_panels(results, metric_name="RMSE", marker_key=None, component_key=None):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()

    for idx, result in enumerate(results):
        ax = axes[idx]
        ax.plot(result["test_t"], result["test_y"], label="observed", color="black")
        ax.plot(result["test_t"], result["forecast"], label="forecast", color="tab:blue")
        ax.plot(result["test_t"], result["test_true"], label="true signal", color="tab:green", alpha=0.7)
        if component_key is not None and component_key in result:
            ax.plot(result["test_t"], result[component_key], label=component_key, color="tab:orange", alpha=0.8)
        if marker_key is not None and marker_key in result:
            marks = result[marker_key]
            for mark in np.atleast_1d(marks):
                ax.axvline(mark, color="tab:red", linestyle="--", alpha=0.6)
        ax.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        ax.grid(True, alpha=0.25)

    axes[-1].plot(range(1, len(results) + 1), [r["metric"] for r in results], marker="o")
    axes[-1].set_xticks(range(1, len(results) + 1))
    axes[-1].set_xlabel("D-rung")
    axes[-1].set_ylabel(metric_name)
    axes[-1].set_title(f"{metric_name} curve")
    axes[-1].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()



## The concept, built once (D1)

A one-split mean-shift detector compares one segment with two:

$$\text{gain}(\tau)=\operatorname{SSE}_{one}-(\operatorname{SSE}_{left}(\tau)+\operatorname{SSE}_{right}(\tau))-\lambda$$

The lesson split has one mean $14.667$, segment means $10.333$ and $19.000$, average squared errors $19.222$ and $0.444$, gain $18.777778$, and CUSUM ending at $24.000$.


In [ ]:

def detect_mean_shift(y, penalty=0.0, min_size=2):
    y = np.asarray(y, dtype=float)
    one_mean = float(np.mean(y))
    one_sse = float(np.sum((y - one_mean) ** 2))
    best = {"split": None, "gain": -np.inf, "left_mean": None, "right_mean": None}

    for split in range(min_size, y.size - min_size + 1):
        left = y[:split]
        right = y[split:]
        left_mean = float(np.mean(left))
        right_mean = float(np.mean(right))
        two_sse = float(np.sum((left - left_mean) ** 2) + np.sum((right - right_mean) ** 2))
        gain = one_sse - two_sse - penalty
        if gain > best["gain"]:
            best = {"split": split, "gain": gain, "left_mean": left_mean, "right_mean": right_mean}

    best["one_mean"] = one_mean
    best["one_sse"] = one_sse
    return best


def cusum_path(y, mu0=10.5, k=0.5):
    s = 0.0
    path = []
    for value in y:
        s = max(0.0, s + float(value) - mu0 - k)
        path.append(s)
    return np.asarray(path)

lesson_y = np.array([10.0, 11.0, 10.0, 18.0, 19.0, 20.0])
lesson = detect_mean_shift(lesson_y, penalty=0.0)
one_avg = lesson["one_sse"] / lesson_y.size
two_avg = one_avg - lesson["gain"]
path = cusum_path(lesson_y, mu0=10.5, k=0.5)

assert np.isclose(lesson["one_mean"], 14.667, atol=5e-4)
assert np.isclose(lesson["left_mean"], 10.333, atol=5e-4)
assert np.isclose(lesson["right_mean"], 19.000, atol=5e-4)
assert np.isclose(one_avg, 19.222, atol=5e-4)
assert np.isclose(two_avg, 0.444, atol=5e-4)
assert np.isclose(lesson["gain"] / lesson_y.size, 18.777778, atol=5e-7)
assert np.isclose(path[-1], 24.000, atol=1e-12)
print("lesson checks passed")



For forecasting, accept a split only if penalized gain is positive. Then refit segment means or slopes after the detected changepoint so forecasts reflect the current regime.


In [ ]:

def segment_refit_forecast(train_t, train_y, test_t, penalty=8.0):
    split_info = detect_mean_shift(train_y, penalty=penalty, min_size=8)
    if split_info["gain"] > 0.0:
        start = split_info["split"]
    else:
        start = 0
    x = np.asarray(train_t[start:], dtype=float)
    y = np.asarray(train_y[start:], dtype=float)
    X = np.column_stack([np.ones_like(x), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    test_X = np.column_stack([np.ones_like(test_t), test_t])
    forecast = test_X @ beta
    return forecast, split_info



## The dataset ladder

We use the same F13 ladder in every notebook so the method faces increasing time-series difficulty inline: D1 constant, D2 drift, D3 drift plus noise, D4 monthly seasonality, and D5 outliers plus a real regime shift.


In [ ]:

rungs = f13_time_series_ladder()
print_ladder_preview(rungs)


In [ ]:

results = []

for rung in rungs:
    train, test = train_test_split_series(rung)
    forecast, split_info = segment_refit_forecast(train["t"], train["y"], test["t"], penalty=8.0)
    metric = rmse(test["true"], forecast)
    if split_info["split"] is None:
        split_time = np.array([])
    elif split_info["gain"] > 0.0:
        split_time = np.array([train["t"][split_info["split"]]])
    else:
        split_time = np.array([])
    results.append({
        "name": rung["name"],
        "test_t": test["t"],
        "test_y": test["y"],
        "test_true": test["true"],
        "forecast": forecast,
        "metric": metric,
        "split_time": split_time,
        "gain": split_info["gain"],
    })

for result in results:
    print(f"{result['name']}: RMSE={result['metric']:.3f}, best gain={result['gain']:.3f}, split marks={result['split_time']}")


In [ ]:

plot_forecast_panels(results, metric_name="RMSE", marker_key="split_time")



## Pitfall on D5: detecting every anomaly as a changepoint

A single spike can reduce SSE locally, but it is not a new regime. The fix is to require penalty validation and sustained evidence before accepting a split.


In [ ]:

d5 = rungs[-1]
train, test = train_test_split_series(d5)
loose = detect_mean_shift(train["y"], penalty=0.0, min_size=2)
strict = detect_mean_shift(train["y"], penalty=20.0, min_size=8)
path = cusum_path(train["y"], mu0=float(np.mean(train["y"][:24])), k=0.5)
print(f"loose split index={loose['split']}, gain={loose['gain']:.3f}")
print(f"strict split index={strict['split']}, gain={strict['gain']:.3f}")
print(f"CUSUM final evidence={path[-1]:.3f}")
print("Strict settings favor sustained evidence over a one-point spike.")



## Evaluate it + Practice

- Report segment-refit RMSE and detection delay; a good split lowers error without firing on one spike.
- Sanity check: a constant D1 series should not receive a positive split after penalty.
- Ablation: set penalty to zero; the detector should over-split noisy or spiky rungs.
- Failure signal: a split exactly at a single outlier rather than before the sustained D5 regime shift.
- Compare CUSUM delay with batch split location to separate online and retrospective use cases.

Practice prompts:
1. Increase the penalty until D3 has no split but D5 still does.


2. Modify the hardest rung and rerun the same metric table.

3. Write one sentence explaining whether RMSE or coverage is the better decision metric here.